<a href="https://colab.research.google.com/github/jac0bmath3w/rail-safety-ai/blob/main/notebooks/05_evaluate_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jac0bmath3w/rail-safety-ai.git

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install llama-index
!pip install -U langchain-text-splitters
!pip install chromadb
!pip install -U bitsandbytes>=0.46.1
!pip install --upgrade transformers
!pip install sentencepiece tiktoken
!pip install rank_bm25
!pip install sentence-transformers

In [ ]:
import os, sys
import torch
from google.colab import drive
from google.colab import userdata
src_path = os.path.join(os.getcwd(), 'rail-safety-ai', 'src')
sys.path.append(src_path)
from vector_store import RailVectorVault
from embed import RailEmbedder
from evaluate import RailAuditJudge
from call_local_llm import run_integrated_audit

# Mount Drive to access Vector DB and save the model
drive.mount('/content/drive')
api_key = userdata.get('GEMINI_API_KEY')
DB_PATH = "/content/drive/MyDrive/rail_ai_project/vector_db"
model_name = "unsloth/Llama-3.2-3B-Instruct"
adapter_path = "/content/drive/MyDrive/rail_ai_project/rail_safety_adapters"

In [ ]:

import importlib
import evaluate
importlib.reload(evaluate)
from evaluate import RailAuditJudge


In [ ]:
embedder = RailEmbedder(model_name='BAAI/bge-base-en-v1.5')
vault = RailVectorVault(embedder_instance=embedder, db_path=DB_PATH)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastLanguageModel.for_inference(model)
model.load_adapter(adapter_path)

In [ ]:
evaluator = RailAuditJudge(audit_function = run_integrated_audit,
                           model = model,
                           tokenizer = tokenizer,
                           vault = vault,
                           api_key = api_key)

In [ ]:
report = evaluator.run_benchmark('/content/rail-safety-ai/data/training/retriever_eval_set_1000_from_csv.json', num_samples=1000, use_dynamic_filter=True)